# Interactive SOH and IC Plotter (NASA MAT Data)

This notebook lets you choose a NASA battery MAT dataset (default: **B0005**) and interactively plot:

- **SOH trend** as a percentage over discharge cycles
- **IC curves** (`-dQ/dV`) for representative selected cycles

It reproduces the `B0005_capacity_trend` style while:

- using **SOH (%)** on the y-axis
- keeping highlighted selected cycles **out of the legend**

## 1. Import Libraries and Configure Widgets

In [22]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.io
import ipywidgets as widgets
from IPython.display import clear_output, display

DATA_DIR = Path("1. BatteryAgingARC-FY08Q4")
DEFAULT_MAT_FILE = "B0005.mat"
DEFAULT_NUM_REP_CYCLES = 5
DEFAULT_DELTA_V = 0.005
DEFAULT_FILTER_WINDOW = 11
DEFAULT_OUTPUT_DIR = Path("demo_outputs_real_data")
PLOT_TEMPLATE = "plotly_white"

mat_files = sorted([p.name for p in DATA_DIR.glob("*.mat")])
if not mat_files:
    raise FileNotFoundError(f"No MAT files found under: {DATA_DIR}")
if DEFAULT_MAT_FILE not in mat_files:
    DEFAULT_MAT_FILE = mat_files[0]

print(f"Found {len(mat_files)} MAT files: {mat_files}")

Found 4 MAT files: ['B0005.mat', 'B0006.mat', 'B0007.mat', 'B0018.mat']


## 2. Load B0005 Capacity Trend and IC Data

The helper functions below read a chosen MAT file, extract discharge cycles, and assemble both:

- per-sample arrays for IC calculations
- tabular cycle-level capacity data for SOH trend plotting

In [23]:
def moving_average(signal: np.ndarray, window: int) -> np.ndarray:
    if window <= 1:
        return signal.copy()
    if window % 2 == 0:
        raise ValueError("Filter window must be an odd integer.")
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(signal, kernel, mode="same")


def load_battery_struct(mat_file: Path, battery_key: str | None = None):
    mat = scipy.io.loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    keys = [k for k in mat.keys() if not k.startswith("__")]

    if battery_key is None:
        stem = mat_file.stem
        if stem in mat:
            battery_key = stem
        elif len(keys) == 1:
            battery_key = keys[0]
        else:
            raise ValueError(
                "Could not infer battery key. Provide battery_key. "
                f"Available keys: {keys}"
            )

    if battery_key not in mat:
        raise KeyError(f"Battery key '{battery_key}' not found in {mat_file.name}.")
    return mat[battery_key], battery_key


def extract_discharge_cycles(battery_struct) -> list[dict[str, object]]:
    raw_cycles = np.atleast_1d(battery_struct.cycle)
    discharge_cycles: list[dict[str, object]] = []

    discharge_idx = 0
    for global_idx, cycle in enumerate(raw_cycles):
        cycle_type = str(getattr(cycle, "type", "")).strip().lower()
        if cycle_type != "discharge":
            continue

        data = getattr(cycle, "data", None)
        if data is None:
            continue

        voltage_v = np.asarray(getattr(data, "Voltage_measured", np.array([])), dtype=float).ravel()
        current_a = np.asarray(getattr(data, "Current_measured", np.array([])), dtype=float).ravel()
        time_s = np.asarray(getattr(data, "Time", np.array([])), dtype=float).ravel()

        length = int(min(voltage_v.size, current_a.size, time_s.size))
        if length < 5:
            continue

        voltage_v = voltage_v[:length]
        current_a = current_a[:length]
        time_s = time_s[:length]

        order = np.argsort(time_s)
        voltage_v = voltage_v[order]
        current_a = current_a[order]
        time_s = time_s[order]

        dt = np.diff(time_s, prepend=time_s[0])
        dt = np.clip(dt, 0.0, None)

        discharge_current_a = np.maximum(-current_a, 0.0)
        capacity_trace_ah = np.cumsum(discharge_current_a * dt) / 3600.0

        capacity_field_ah = float(getattr(data, "Capacity", np.nan))
        capacity_scalar_ah = (
            capacity_field_ah if np.isfinite(capacity_field_ah) else float(capacity_trace_ah[-1])
        )

        discharge_cycles.append(
            {
                "global_index": int(global_idx),
                "discharge_index": int(discharge_idx),
                "time_s": time_s,
                "voltage_v": voltage_v,
                "current_a": current_a,
                "capacity_trace_ah": capacity_trace_ah,
                "capacity_end_ah": float(capacity_trace_ah[-1]),
                "capacity_field_ah": capacity_field_ah,
                "capacity_scalar_ah": float(capacity_scalar_ah),
            }
        )
        discharge_idx += 1

    if not discharge_cycles:
        raise RuntimeError("No discharge cycles were found in this dataset.")
    return discharge_cycles


def build_capacity_dataframe(discharge_cycles: list[dict[str, object]]) -> pd.DataFrame:
    records = [
        {
            "cycle": int(c["discharge_index"]) + 1,
            "capacity_ah": float(c["capacity_scalar_ah"]),
            "capacity_source": (
                "field"
                if np.isfinite(float(c["capacity_field_ah"]))
                else "integrated_fallback"
            ),
        }
        for c in discharge_cycles
    ]
    capacity_df = pd.DataFrame(records)

    required_columns = {"cycle", "capacity_ah"}
    missing = required_columns - set(capacity_df.columns)
    if missing:
        raise ValueError(f"Capacity dataframe missing required columns: {sorted(missing)}")

    return capacity_df

## 3. Prepare Data and Compute SOH (%)

SOH is computed as:

\[
\text{SOH}(\%) = 100 \times \frac{C_{\text{cycle}}}{C_{\text{ref}}}
\]

By default, this notebook uses the first valid cycle capacity as \(C_{\text{ref}}\).

In [24]:
def compute_soh_dataframe(
    capacity_df: pd.DataFrame,
    reference_capacity_ah: float | None = None,
) -> tuple[pd.DataFrame, float]:
    required_columns = {"cycle", "capacity_ah"}
    missing = required_columns - set(capacity_df.columns)
    if missing:
        raise ValueError(f"Cannot compute SOH, missing columns: {sorted(missing)}")

    valid_capacity = capacity_df["capacity_ah"].dropna()
    valid_capacity = valid_capacity[np.isfinite(valid_capacity)]
    if valid_capacity.empty:
        raise ValueError("No finite capacity values available for SOH computation.")

    if reference_capacity_ah is None or reference_capacity_ah <= 0:
        c_ref = float(valid_capacity.iloc[0])
    else:
        c_ref = float(reference_capacity_ah)

    soh_df = capacity_df.copy()
    soh_df["soh_percent"] = 100.0 * soh_df["capacity_ah"] / c_ref
    return soh_df, c_ref


def pick_representative_indices(total_count: int, n_select: int) -> list[int]:
    if total_count <= 0:
        return []

    n = max(1, min(int(n_select), total_count))
    raw_idx = [int(round(x)) for x in np.linspace(0, total_count - 1, n)]

    selected: list[int] = []
    for idx in raw_idx:
        if idx not in selected:
            selected.append(idx)

    if len(selected) < n:
        for idx in range(total_count):
            if idx not in selected:
                selected.append(idx)
            if len(selected) == n:
                break

    return selected

## 4. Create SOH Trend Plot for B0005 (Hide Selected-Cycle Legend Entries)

The highlighted selected-cycle markers are shown on the figure, but intentionally hidden from the legend.

In [25]:
def create_soh_trend_figure(
    soh_df: pd.DataFrame,
    selected_cycle_numbers: list[int],
    battery_key: str,
) -> go.Figure:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=soh_df["cycle"],
            y=soh_df["soh_percent"],
            mode="lines",
            name="SOH trend",
            line={"color": "#005f73", "width": 2.2},
        )
    )

    selected_df = soh_df[soh_df["cycle"].isin(selected_cycle_numbers)]
    if not selected_df.empty:
        fig.add_trace(
            go.Scatter(
                x=selected_df["cycle"],
                y=selected_df["soh_percent"],
                mode="markers",
                name="Selected cycles",
                showlegend=False,
                marker={"color": "#bb3e03", "size": 8, "line": {"width": 0.8, "color": "#701f00"}},
                hovertemplate="Cycle %{x}<br>SOH %{y:.2f}%<extra></extra>",
            )
        )

    fig.update_layout(
        title=f"{battery_key}_capacity_trend",
        template=PLOT_TEMPLATE,
        height=430,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        margin={"l": 60, "r": 30, "t": 70, "b": 60},
    )
    fig.update_xaxes(title_text="Discharge cycle number", showgrid=True)
    fig.update_yaxes(title_text="SOH (%)", showgrid=True)
    return fig

## 5. Create Interactive IC Plot for Selected Cycles

IC curves are computed as \(-dQ/dV\) on a fixed voltage grid for representative cycles in the selected cycle range.

In [26]:
def compute_ic_curve(
    voltage_v: np.ndarray,
    capacity_trace_ah: np.ndarray,
    delta_v: float,
) -> tuple[np.ndarray, np.ndarray]:
    valid = np.isfinite(voltage_v) & np.isfinite(capacity_trace_ah)
    v = voltage_v[valid]
    q = capacity_trace_ah[valid]

    if v.size < 5:
        return np.array([]), np.array([])

    v_monotonic = np.minimum.accumulate(v)
    v_asc = v_monotonic[::-1]
    q_asc = q[::-1]

    v_unique, unique_idx = np.unique(v_asc, return_index=True)
    q_unique = q_asc[unique_idx]

    if v_unique.size < 5:
        return np.array([]), np.array([])

    v_low = float(v_unique.min())
    v_high = float(v_unique.max())
    if (v_high - v_low) < (2.0 * delta_v):
        return np.array([]), np.array([])

    v_grid = np.arange(v_low, v_high + delta_v, delta_v)
    q_grid = np.interp(v_grid, v_unique, q_unique)

    dq = np.diff(q_grid)
    dv = np.diff(v_grid)
    ic = -(dq / np.maximum(dv, 1e-12))
    v_mid = 0.5 * (v_grid[1:] + v_grid[:-1])
    return v_mid, ic


def create_ic_figure(
    selected_cycles: list[dict[str, object]],
    selected_indices: list[int],
    delta_v: float,
    filter_window: int,
    battery_key: str,
) -> go.Figure:
    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        subplot_titles=("Raw IC Curves", f"Filtered IC Curves (MA window={filter_window})"),
        horizontal_spacing=0.09,
    )

    colors = [
        "#0a9396",
        "#94d2bd",
        "#ee9b00",
        "#ca6702",
        "#bb3e03",
        "#9b2226",
        "#6a4c93",
        "#5f0f40",
    ]

    for rank, cycle_idx in enumerate(selected_indices):
        cycle = selected_cycles[cycle_idx]
        cycle_number = int(cycle["discharge_index"]) + 1
        v_ic, ic_raw = compute_ic_curve(
            np.asarray(cycle["voltage_v"]),
            np.asarray(cycle["capacity_trace_ah"]),
            delta_v,
        )
        if v_ic.size == 0:
            continue

        ic_filtered = moving_average(ic_raw, filter_window) if ic_raw.size else ic_raw
        color = colors[rank % len(colors)]
        trace_name = f"Discharge #{cycle_number}"

        fig.add_trace(
            go.Scatter(
                x=v_ic,
                y=ic_raw,
                mode="lines",
                name=trace_name,
                legendgroup=trace_name,
                line={"color": color, "width": 1.5},
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=v_ic,
                y=ic_filtered,
                mode="lines",
                name=trace_name,
                legendgroup=trace_name,
                showlegend=False,
                line={"color": color, "width": 2.2},
            ),
            row=1,
            col=2,
        )

    fig.update_layout(
        title=f"{battery_key}: IC Curves for Representative Cycles",
        template=PLOT_TEMPLATE,
        height=500,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
        margin={"l": 60, "r": 30, "t": 70, "b": 60},
    )
    fig.update_xaxes(title_text="Voltage V (V)", showgrid=True, row=1, col=1)
    fig.update_xaxes(title_text="Voltage V (V)", showgrid=True, row=1, col=2)
    fig.update_yaxes(title_text="IC = -dQ/dV (Ah/V)", showgrid=True, row=1, col=1)
    fig.update_yaxes(showgrid=True, row=1, col=2)
    return fig

## 6. Wire Up Dataset and Cycle Controls

Use the controls below to choose dataset, cycle range, representative cycle count, and IC smoothing settings.

In [27]:
dataset_dropdown = widgets.Dropdown(
    options=mat_files,
    value=DEFAULT_MAT_FILE,
    description="MAT file",
    layout=widgets.Layout(width="320px"),
)
battery_key_text = widgets.Text(
    value="",
    placeholder="Optional (auto if blank)",
    description="Battery key",
    layout=widgets.Layout(width="320px"),
)
cycle_range_slider = widgets.IntRangeSlider(
    value=(1, 1),
    min=1,
    max=1,
    step=1,
    description="Cycle range",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
num_cycles_slider = widgets.IntSlider(
    value=DEFAULT_NUM_REP_CYCLES,
    min=1,
    max=20,
    step=1,
    description="# IC cycles",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
delta_v_slider = widgets.FloatSlider(
    value=DEFAULT_DELTA_V,
    min=0.001,
    max=0.02,
    step=0.001,
    readout_format=".3f",
    description="Delta V",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
filter_window_slider = widgets.IntSlider(
    value=DEFAULT_FILTER_WINDOW,
    min=3,
    max=41,
    step=2,
    description="MA window",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
save_png_checkbox = widgets.Checkbox(
    value=False,
    description="Save PNG outputs",
)
output_dir_text = widgets.Text(
    value=str(DEFAULT_OUTPUT_DIR),
    description="Output dir",
    layout=widgets.Layout(width="420px"),
)
run_button = widgets.Button(
    description="Plot SOH + IC",
    button_style="primary",
    icon="line-chart",
)
ui_out = widgets.Output()


def refresh_cycle_range() -> None:
    mat_file = DATA_DIR / dataset_dropdown.value
    battery_key_input = battery_key_text.value.strip() or None

    try:
        battery_struct, _ = load_battery_struct(mat_file, battery_key_input)
        discharge_cycles = extract_discharge_cycles(battery_struct)
        total_cycles = len(discharge_cycles)
    except Exception:
        total_cycles = 1

    total_cycles = max(1, total_cycles)
    cycle_range_slider.min = 1
    cycle_range_slider.max = total_cycles
    cycle_range_slider.value = (1, total_cycles)


def run_dashboard(_=None) -> None:
    with ui_out:
        clear_output(wait=True)

        try:
            mat_file = DATA_DIR / dataset_dropdown.value
            battery_key_input = battery_key_text.value.strip() or None

            battery_struct, battery_key = load_battery_struct(mat_file, battery_key_input)
            discharge_cycles = extract_discharge_cycles(battery_struct)
            capacity_df = build_capacity_dataframe(discharge_cycles)
            soh_df, c_ref = compute_soh_dataframe(capacity_df)

            start_cycle, end_cycle = cycle_range_slider.value
            cycle_pool = [
                c
                for c in discharge_cycles
                if start_cycle <= (int(c["discharge_index"]) + 1) <= end_cycle
            ]
            if not cycle_pool:
                raise ValueError("No cycles are available in the selected range.")

            selected_local_indices = pick_representative_indices(
                len(cycle_pool), num_cycles_slider.value
            )
            selected_cycle_numbers = [
                int(cycle_pool[i]["discharge_index"]) + 1 for i in selected_local_indices
            ]

            soh_fig = create_soh_trend_figure(soh_df, selected_cycle_numbers, battery_key)
            ic_fig = create_ic_figure(
                selected_cycles=cycle_pool,
                selected_indices=selected_local_indices,
                delta_v=float(delta_v_slider.value),
                filter_window=int(filter_window_slider.value),
                battery_key=battery_key,
            )

            display(soh_fig)
            display(ic_fig)

            print(f"Battery key: {battery_key}")
            print(f"Reference capacity for SOH: {c_ref:.3f} Ah")
            print(f"Total discharge cycles: {len(discharge_cycles)}")
            print(f"Selected cycles for IC: {selected_cycle_numbers}")

            if save_png_checkbox.value:
                output_dir = Path(output_dir_text.value).expanduser()
                output_dir.mkdir(parents=True, exist_ok=True)

                soh_path = output_dir / f"{battery_key}_capacity_trend.png"
                ic_path = output_dir / f"{battery_key}_ic_raw_vs_filtered.png"

                try:
                    soh_fig.write_image(str(soh_path), scale=2)
                    ic_fig.write_image(str(ic_path), scale=2)
                    print("\nSaved PNG files:")
                    print(f"  - {soh_path}")
                    print(f"  - {ic_path}")
                except Exception as exc:
                    soh_html = soh_path.with_suffix(".html")
                    ic_html = ic_path.with_suffix(".html")
                    soh_fig.write_html(str(soh_html))
                    ic_fig.write_html(str(ic_html))
                    print("\nPNG export failed. Install kaleido for static images:")
                    print("  pip install kaleido")
                    print(f"Reason: {exc}")
                    print("Saved HTML fallback files:")
                    print(f"  - {soh_html}")
                    print(f"  - {ic_html}")

        except Exception as exc:
            print(f"Error: {exc}")


def _on_dataset_change(change):
    if change["name"] == "value":
        refresh_cycle_range()


dataset_dropdown.observe(_on_dataset_change, names="value")

controls_left = widgets.VBox(
    [
        dataset_dropdown,
        battery_key_text,
        cycle_range_slider,
        num_cycles_slider,
    ]
)
controls_right = widgets.VBox(
    [
        delta_v_slider,
        filter_window_slider,
        output_dir_text,
        save_png_checkbox,
        run_button,
    ]
)
controls = widgets.HBox([controls_left, controls_right])

run_button.on_click(run_dashboard)
refresh_cycle_range()
display(controls)
display(ui_out)

Output()

## 7. Add Reusable Run Cell and Export Notes

Run the next cell to render with current controls (default dataset is B0005). If PNG export fails, install `kaleido` to enable Plotly static image saving.

In [28]:
# Reusable run cell.
# Tip: enable "Save PNG outputs" above if you want to export files.
run_dashboard()